# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata_json = json.loads(dataset.metadata.to_json())
print(f"{metadata_json['name']}: {metadata_json['description']}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# Access metadata JSON-LD directly
from pprint import pprint

# Get record sets and their field structure
print("Available record sets (by @id):")
record_sets = dataset.metadata.record_sets
for rs in record_sets:
    print(f"  Record Set: {rs['@id']}  |  name: {rs.get('name', '')}")
    print("    Fields and their @id's:")
    for field in rs.get('fields', []):
        print(f"      - {field['@id']} (name: {field.get('name', '')}, type: {field.get('dataType', '')})")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract all available record sets
record_sets = [rs['@id'] for rs in dataset.metadata.record_sets]
dataframes = {}

for record_set_id in record_sets:
    print(f"Loading records for record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Columns in record set '{record_set_id}':\n  {df.columns.tolist()}\n")
        print(df.head())
    else:
        print(f"No records found for record set {record_set_id}\n")

# For demonstration, pick the first record set containing data for further EDA
if dataframes:
    main_record_set_id = list(dataframes.keys())[0]
    print(f"Selected record set for EDA: {main_record_set_id}")
    print(f"Sample data:\n{dataframes[main_record_set_id].head()}")
else:
    main_record_set_id = None

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
import numpy as np
import warnings
warnings.filterwarnings('ignore')

if main_record_set_id:
    df = dataframes[main_record_set_id]
    # Identify numeric fields by dtype (could also use field metadata)
    numeric_fields = df.select_dtypes(include=[np.number]).columns.tolist()
    print(f"Numeric fields found: {numeric_fields}")

    if numeric_fields:
        numeric_field = numeric_fields[0]
        print(f"Analyzing numeric field: {numeric_field}")
        threshold = df[numeric_field].mean() if not df[numeric_field].isnull().all() else None
        # Use a simple filter: field > mean
        if threshold is not None:
            filtered_df = df[df[numeric_field] > threshold]
            print(f"Filtered records where {numeric_field} > {threshold:.2f}:")
            print(filtered_df.head())

            # Normalization
            filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
            print(f"Normalized \"{numeric_field}\" for filtered records:")
            print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

            # Find a categorical field for grouping (object type, not the numeric_field)
            group_field_candidates = df.select_dtypes(include=[object]).columns.tolist()
            group_field = None
            for col in group_field_candidates:
                if col != numeric_field and df[col].nunique() < max(20, df.shape[0]//10):
                    group_field = col
                    break
            if group_field:
                grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
                print(f"Grouped data by '{group_field}':")
                print(grouped_df.head())
            else:
                print("No appropriate group field found for grouping.")
        else:
            print("All values in numeric field are missing. Skipping filtering and normalization.")
    else:
        print("No numeric fields found in the selected record set.")
else:
    print("No data available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and dataframes[main_record_set_id].shape[0] > 0:
    df = dataframes[main_record_set_id]
    # If we identified a numeric field above, plot its distribution
    if 'numeric_field' in locals() and numeric_field in df.columns:
        plt.figure(figsize=(8,4))
        sns.histplot(df[numeric_field].dropna(), kde=True)
        plt.title(f'Distribution of {numeric_field}')
        plt.xlabel(numeric_field)
        plt.ylabel('Count')
        plt.show()

    # Scatter plot of two numeric fields, if available
    if len(df.select_dtypes(include=[np.number]).columns) >= 2:
        num_cols = df.select_dtypes(include=[np.number]).columns[:2]
        plt.figure(figsize=(6,5))
        sns.scatterplot(x=df[num_cols[0]], y=df[num_cols[1]])
        plt.title(f"Scatter plot: {num_cols[0]} vs {num_cols[1]}")
        plt.xlabel(num_cols[0])
        plt.ylabel(num_cols[1])
        plt.show()
else:
    print("No data available for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

<!-- Summary observations go here -->
In this notebook, we:
- Loaded FAIR^2 Croissant formatted mass spectrometry data on extracellular vesicle proteins using `mlcroissant`
- Explored record set and field structure by `@id`
- Extracted and viewed tabular data from each record set
- Performed basic EDA and visualization (numeric field filtering, normalization, and grouping where possible)
- Identified that further analysis requires domain context, i.e., deciding which features and samples are most salient for biological study.

For more information, review the Croissant schema at [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) and consult the dataset documentation.